In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import runpy
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from trompy import save_figure_atomic

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pixi.toml").exists())
globals().update(runpy.run_path(str(ROOT / "src" / "repo_paths.py")))

from pickles_to_snips import (assemble_snips_and_metadata,
                           get_snips_and_xarray,
                           mean_snips_by_group,
                           reduce_to_matched_mice,
                           calculate_grain_sucrose_difference
)

from figure_plotting import (FIG4_DIVERGING_CMAP,
                             get_palette,
                             make_heatplot_and_lineplot_per_session,
                             make_heatplot_and_lineplot_per_session_mean_of_rats,
                             make_difference_plot,
                             make_auc_plot
)

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["savefig.transparent"] = True

cmap_diverging = FIG4_DIVERGING_CMAP

In [ ]:
# Palette options for NR/PR x grain/sucrose
# Structure:
# option["NR"]["grain"], option["NR"]["sucrose"], option["PR"]["grain"], option["PR"]["sucrose"]

ACTIVE_PALETTE_NAME = "option_4_grey_reddish"
ACTIVE_PALETTE, COLORS_NR, COLORS_PR, PHASE_PELLET_COLORS = get_palette(ACTIVE_PALETTE_NAME)

print("Using:", ACTIVE_PALETTE_NAME)
print("NR colors [grain, sucrose]:", COLORS_NR)
print("PR colors [grain, sucrose]:", COLORS_PR)

In [ ]:
DATAFOLDER = DATA_DIR
FIGSFOLDER = FIGS_DIR / "panels"
FIGSFOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
# for testing
filename = DATAFOLDER / "W1_1_NR_Grain_FR1_ttl_pellet_onset.pickle"
snips, x_array = get_snips_and_xarray(filename, "grain", "NR1")

In [ ]:
session_specs = [
    {"filename": DATAFOLDER / "W1_1_NR_Grain_FR1_ttl_pellet_offset.pickle", "pellettype": "grain", "dietphase": "NR1"},
    {"filename": DATAFOLDER / "W2_1_PR_Grain_FR1_ttl_pellet_offset.pickle", "pellettype": "grain", "dietphase": "PR"},
    {"filename": DATAFOLDER / "W3_1_NR_Grain_FR1_ttl_pellet_offset.pickle", "pellettype": "grain", "dietphase": "NR2"},
    {"filename": DATAFOLDER / "W1_3_NR_Sucrose_FR1_ttl_pellet_offset.pickle", "pellettype": "sucrose", "dietphase": "NR1"},
    {"filename": DATAFOLDER / "W2_3_PR_Sucrose_FR1_ttl_pellet_offset.pickle", "pellettype": "sucrose", "dietphase": "PR"},
    {"filename": DATAFOLDER / "W3_3_NR_Sucrose_FR1_ttl_pellet_offset.pickle", "pellettype": "sucrose", "dietphase": "NR2"},
]

snips, x_array = assemble_snips_and_metadata(session_specs, nbins=150)
phases = ["NR1", "PR", "NR2"]

In [ ]:
x_array.columns

In [ ]:
(x_array
 .query("id == 'FOTOA07'")
 .loc[:, ["id", "dietphase", "pellettype", "max_pellets", "min_pellets_by_pellettype"]]
)

In [ ]:
VMIN, VMAX = -2, 2
MATCHING_ON = True  # set False to disable grain/sucrose ID matching within phase
EXTRA_QUERY = "pellet_number < 6 and max_pellets > 5"

x_reduced, snips_red, match_summary = reduce_to_matched_mice(
    x_array,
    snips,
    EXTRA_QUERY,
    require_both_pellet_types=MATCHING_ON,
 )

print("Matching enabled:", MATCHING_ON)
print("Reduced rows:", len(x_reduced))
print("Unique mice by phase and pellet after filtering:")
print(x_reduced.groupby(["dietphase", "pellettype"])["id"].nunique())
print("\nID summary (before vs after):")
print(match_summary)

## NR1
df, tr_grain = mean_snips_by_group(x_reduced, snips_red, group_cols=["id", "dietphase", "pellettype"],
                             dietphase="NR1", pellettype="grain", extra_query=EXTRA_QUERY)

df, tr_suc = mean_snips_by_group(x_reduced, snips_red, group_cols=["id", "dietphase", "pellettype"],
                             dietphase="NR1", pellettype="sucrose", extra_query=EXTRA_QUERY)

f, ax, _ = make_heatplot_and_lineplot_per_session_mean_of_rats(tr_grain, tr_suc, vmin=VMIN, vmax=VMAX, cmap=cmap_diverging,
                                                               colors=COLORS_NR)
save_figure_atomic(f, "figSx_fr1_photo_p_offset_NR1", FIGSFOLDER)

## PR
df, tr_grain = mean_snips_by_group(x_reduced, snips_red, group_cols=["id", "dietphase", "pellettype"],
                             dietphase="PR", pellettype="grain", extra_query=EXTRA_QUERY)

df, tr_suc = mean_snips_by_group(x_reduced, snips_red, group_cols=["id", "dietphase", "pellettype"],
                             dietphase="PR", pellettype="sucrose", extra_query=EXTRA_QUERY)

f, ax, _ = make_heatplot_and_lineplot_per_session_mean_of_rats(tr_grain, tr_suc, vmin=VMIN, vmax=VMAX, cmap=cmap_diverging,
                                                               colors=COLORS_PR)
save_figure_atomic(f, "figSx_fr1_photo_p_offset_PR", FIGSFOLDER)

## NR2
df, tr_grain = mean_snips_by_group(x_reduced, snips_red, group_cols=["id", "dietphase", "pellettype"],
                             dietphase="NR2", pellettype="grain", extra_query=EXTRA_QUERY)

df, tr_suc = mean_snips_by_group(x_reduced, snips_red, group_cols=["id", "dietphase", "pellettype"],
                             dietphase="NR2", pellettype="sucrose", extra_query=EXTRA_QUERY)

f, ax, f_cbar = make_heatplot_and_lineplot_per_session_mean_of_rats(tr_grain, tr_suc, vmin=VMIN, vmax=VMAX, cmap=cmap_diverging,
                                                                    colors=COLORS_NR)
save_figure_atomic(f, "figSx_fr1_photo_p_offset_NR2", FIGSFOLDER)
save_figure_atomic(f_cbar, "figSx_fr1_photo_p_offset_cbar", FIGSFOLDER)

In [ ]:
# make AUC graph

df, tr = mean_snips_by_group(x_reduced, snips_red, group_cols=["id", "dietphase", "pellettype"],
                             extra_query=EXTRA_QUERY)
auc = np.mean(tr[:, 50:100], axis=1)
auc_df = df.assign(auc=auc)

# Barplot of AUC by diet phase and pellet type, with individual-rat jittered points
fig, ax = make_auc_plot(auc_df, colors_nr=COLORS_NR, colors_pr=COLORS_PR)

save_figure_atomic(fig, "figSx_fr1_photo_p_offset_auc", FIGSFOLDER)

# Companion graph: difference of per-rat means (grain mean - sucrose mean) by diet phase
diff_stats = calculate_grain_sucrose_difference(auc_df)

fig, ax = make_difference_plot(diff_stats, colors=[COLORS_NR[0], COLORS_PR[0], COLORS_NR[0]])
ax.set_ylim(-0.5, 1.5)

save_figure_atomic(fig, "figSx_fr1_photo_p_offset_auc_ratio", FIGSFOLDER)

In [ ]:
# Compare grain-fraction ratios to 0.5 for each diet phase
# Primary test: Wilcoxon signed-rank (non-parametric, paired to 0.5)
# Reference test: one-sample t-test
from scipy import stats

diff_points = diff_stats["diff_points"]

diff_test_rows = []
alpha = 0.05
n_phases = len(phases)

for phase in phases:
    vals = diff_points.groupby("dietphase")["diff"].get_group(phase).to_numpy() if phase in diff_points.index.get_level_values("dietphase") else np.array([])
    vals = vals[~np.isnan(vals)]
    n = len(vals)

    if n == 0:
        diff_test_rows.append({
            "dietphase": phase,
            "n": 0,
            "mean_diff": np.nan,
            "median_diff": np.nan,
            "wilcoxon_stat": np.nan,
            "wilcoxon_p": np.nan,
            "ttest_t": np.nan,
            "ttest_p": np.nan,
        })
        continue

    diffs = vals - 0

    # Wilcoxon can fail when all diffs are exactly zero; report NaN in that edge case
    try:
        wres = stats.wilcoxon(diffs, alternative="two-sided", zero_method="wilcox", correction=False)
        w_stat, w_p = float(wres.statistic), float(wres.pvalue)
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    tres = stats.ttest_1samp(vals, popmean=0, nan_policy="omit")
    t_stat, t_p = float(tres.statistic), float(tres.pvalue)

    diff_test_rows.append({
        "dietphase": phase,
        "n": int(n),
        "mean_diff": float(np.mean(vals)),
        "median_diff": float(np.median(vals)),
        "wilcoxon_stat": w_stat,
        "wilcoxon_p": w_p,
        "ttest_t": t_stat,
        "ttest_p": t_p,
    })

diff_tests = pd.DataFrame(diff_test_rows)

# Simple multiplicity control across phases (Bonferroni)
diff_tests["wilcoxon_p_bonf"] = np.minimum(diff_tests["wilcoxon_p"] * n_phases, 1.0)
diff_tests["ttest_p_bonf"] = np.minimum(diff_tests["ttest_p"] * n_phases, 1.0)

diff_tests["wilcoxon_sig_0p05"] = diff_tests["wilcoxon_p_bonf"] < alpha
diff_tests["ttest_sig_0p05"] = diff_tests["ttest_p_bonf"] < alpha

print("Per-phase tests vs 0 (two-sided):")
display(diff_tests)



In [ ]:
## trial-by-trial
# 
VMIN, VMAX = -2, 2
f, ax, ax_cbar = make_heatplot_and_lineplot_per_session(snips,
                                                        x_array,
                                                        session="NR1", 
                                                        max_pellets_filter=6,
                                                        vmin=VMIN, vmax=VMAX)

f, ax, ax_cbar = make_heatplot_and_lineplot_per_session(snips,
                                                        x_array,
                                                        session="PR", 
                                                        max_pellets_filter=6,
                                                        vmin=VMIN, vmax=VMAX)

f, ax, ax_cbar = make_heatplot_and_lineplot_per_session(snips,
                                                        x_array,
                                                        session="NR2", 
                                                        max_pellets_filter=6,
                                                        vmin=VMIN, vmax=VMAX)